## Родительская схема

Определяет базово поведение всех дочерних схем (по сути написали её один раз и больше не трогаем, иногда модифицируем когда оч необходимо)

In [1]:
import pandera as pa
import geopandas as gpd
from pandera.typing import Index
from pandera.typing.geopandas import GeoSeries
from shapely import Point, MultiPoint, Polygon, MultiPolygon, LineString, MultiLineString

class BaseSchema(pa.DataFrameModel):
    idx: Index[int] = pa.Field(unique=True)
    geometry: GeoSeries
    _geom_types = [Point, MultiPoint, Polygon, MultiPolygon, LineString, MultiLineString]

    class Config:
        strict = "filter"
        add_missing_columns = True

    @classmethod
    def to_gdf(cls):
        columns = cls.to_schema().columns.keys()
        return gpd.GeoDataFrame(data=[], columns=columns, crs=4326)

    @pa.check("geometry")
    @classmethod
    def check_geometry(cls, series):
        return series.map(lambda x: any([isinstance(x, geom_type) for geom_type in cls._geom_types]))

## Дочерняя схема

Пример того, как можно используя родительскую схему описать свой формат данных

In [7]:
from pandera.typing import Series

class ZonesSchema(BaseSchema):

    _geom_types = [Polygon, MultiPolygon]
    zone: Series[str]
    column: Series[int] = pa.Field(ge=0, default=123)

## Проверка самого ГДФ

Можно делать под капотом, вернет проверенный гдф

In [17]:
# создание гдф

gdf = gpd.GeoDataFrame(data=[
  {
    'zone': 'blabla',
    'geometry': Polygon([Point(0,0), Point(0,1), Point(1,1)]),
  }
])

In [18]:
# проверка

gdf = ZonesSchema(gdf)
gdf

,zone,geometry,column
0,blabla,"POLYGON ((0.00000 0.00000, 0.00000 1.00000, 1....",123


## Твои схемки и валидация

### Импорты и загрузка данных

In [4]:
import sys
sys.path.append('../')
import transport_frames.src.graph_builder.criteria as criteria
import transport_frames.src.metrics.indicators as indicators
import transport_frames.src.metrics.grade_territory as grade_territory
import transport_frames.src.graph_builder.graphbuilder as graphbuilder
import momepy
import osmnx as ox
import geopandas as gpd
import shapely
import pandas as pd
import networkx as nx
import numpy as np
import json

/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
region_name = 'Ленинградская_область'
local_crs = 32636
geocode = 176095
region_capital = ox.geocode_to_gdf('N27025179',by_osmid=True)
city = ox.geocode_to_gdf(f'R{geocode}', by_osmid=True).to_crs(epsg=4326)
city['layer'] = region_name
city['status'] = 'region'

# Считывание входных данных

PLACEHOLDER = gpd.GeoDataFrame(geometry=[]) # если нет данных о сервисе

# полигоны районов и муниципальных образований
settlement = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_settlement.geojson')
district = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_district.geojson')

# населенные пункты и админ.центры точками
region_points = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_region_points.geojson')
admin_centers = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_district_admin_centres_18_nodes.geojson')
# либо:
# admin_centers = region_points[region_points['is_admin_centre_district']==True]


# Сервисы
railway_stations = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_railway_stations.geojson')
railway_roads = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_railway_roads.geojson')
bus_stops = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_bus_stops.geojson')
bus_routes = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_bus_routes.geojson')
fuel_stations = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_fuel_stations.geojson')
# ferry_terminal = PLACEHOLDER
ferry_terminal = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_ferry_terminal.geojson')
local_aerodrome = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_local_aerodrome.geojson')
international_aerodrome = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_international_aerodrome.geojson')
# либо:
# international_aerodrome = local_aerodrome[local_aerodrome['aerodrome:type']=='international']
# international_aerodrome['geometry'] = shapely.centroid(international_aerodrome['geometry']).set_crs(international_aerodrome.crs)

# Природные объекты
water = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_water.geojson')
nature_reserve = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/{region_name}/{region_name}_nature_reserve.geojson')

# Кастомная территория
custom_territory = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Ленинградская_область/project Светогорского поселения.geojson')
custom_territory_center = custom_territory.geometry.representative_point()
custom_territory_center = gpd.GeoDataFrame([{'geometry': custom_territory_center.iloc[0]}], crs=custom_territory_center.crs).to_crs(local_crs)

# Полигон РФ, необходим для построения транспортного каркаса
russia = ox.geocode_to_gdf("Russia") 
regions = gpd.read_file(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/regions_of_russia.geojson') #  get regions
regions = regions[regions['ISO3166-2']!='RU-CHU']
regions = regions.to_crs(city.crs)

In [6]:
citygraph = nx.read_graphml(f'/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/graphs/{region_name}/{region_name}_uds.graphml')
citygraph = graphbuilder.convert_list_attr_from_str(citygraph)
citygraph = indicators.prepare_graph(citygraph)

### Валидация для graphbuilder

In [7]:
from pandera.typing import Series

In [7]:
settlement # переименовать сущности

,osm_id,boundary,admin_level,name,layer,2019,2020,2021,2022,2023,key,admiin center,status,geometry
0,-7770974,administrative,8,Ощепковское,Абатский район,1864.0,1819.0,1785.0,1736.0,2208.0,None,None,Сельское поселение,"MULTIPOLYGON (((70.35363 56.53420, 70.41604 56..."
1,-7770973,administrative,8,Назаровское,Абатский район,213.0,200.0,185.0,172.0,377.0,None,None,Сельское поселение,"MULTIPOLYGON (((70.72792 56.52697, 70.72904 56..."
2,-5509083,administrative,8,Шевыринское,Абатский район,895.0,870.0,845.0,816.0,996.0,None,None,Сельское поселение,"MULTIPOLYGON (((70.32875 56.18267, 70.32965 56..."
3,-5509082,administrative,8,Тушнолобовское,Абатский район,1085.0,1074.0,1051.0,1034.0,1157.0,None,None,Сельское поселение,"MULTIPOLYGON (((69.89056 56.18638, 69.89624 56..."
4,-5509081,administrative,8,Партизанское,Абатский район,402.0,384.0,370.0,351.0,390.0,None,None,Сельское поселение,"MULTIPOLYGON (((70.67055 56.08794, 70.67063 56..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,-1711441,administrative,6,Тобольск,Городской округ Тобольск,102242.0,102279.0,102071.0,101401.0,103175.0,71410000000,None,Городской округ,"MULTIPOLYGON (((68.14574 58.18468, 68.14770 58..."
276,-897992,administrative,6,Тюмень,Городской округ Тюмень,788666.0,807271.0,816700.0,828575.0,855618.0,71401000000,None,Городской округ,"MULTIPOLYGON (((65.26049 57.28383, 65.26101 57..."
277,-1711462,administrative,6,Ялуторовск,городской округ Ялуторовск,39919.0,40273.0,39947.0,39822.0,38882.0,71415000000,None,Городской округ,"MULTIPOLYGON (((66.21807 56.64639, 66.22329 56..."
278,-1711469,administrative,6,Заводоуковский,Заводоуковский городской округ,46587.0,46768.0,46382.0,46173.0,47461.0,71222000000,г Заводоуковск,Городской округ,"MULTIPOLYGON (((66.08974 56.48680, 66.10479 56..."


In [19]:
city.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   geometry      1 non-null      geometry
 1   bbox_north    1 non-null      float64 
 2   bbox_south    1 non-null      float64 
 3   bbox_east     1 non-null      float64 
 4   bbox_west     1 non-null      float64 
 5   place_id      1 non-null      int64   
 6   osm_type      1 non-null      object  
 7   osm_id        1 non-null      int64   
 8   lat           1 non-null      float64 
 9   lon           1 non-null      float64 
 10  class         1 non-null      object  
 11  type          1 non-null      object  
 12  place_rank    1 non-null      int64   
 13  importance    1 non-null      float64 
 14  addresstype   1 non-null      object  
 15  name          1 non-null      object  
 16  display_name  1 non-null      object  
 17  layer         1 non-null      object  
 18  status

In [23]:
# класс валидации полигона для графа

class GraphbuilderSchema(BaseSchema):
    
    name: Series[str]
    layer: Series[str]
    status: Series[str]
    _geom_types = [Polygon, MultiPolygon]

In [24]:
city = GraphbuilderSchema(city)
city

,geometry,name,layer,status
0,"MULTIPOLYGON (((26.44000 59.93833, 26.62887 59...",Leningrad Oblast,Ленинградская_область,region


### Валидация для indicators_area

In [3]:
district.info()

NameError: name 'district' is not defined

In [ ]:
# класс валидации полигонов территории

class IndicatorsSchema(BaseSchema):

    name: Series[str]
    layer: Series[str]
    status: Series[str]
    _geom_types = [Polygon, MultiPolygon]

In [ ]:
city = IndicatorsSchema(city)
city

,geometry,name,layer,status
0,"MULTIPOLYGON (((26.44000 59.93833, 26.62887 59...",Leningrad Oblast,Ленинградская_область,region


In [ ]:
district = IndicatorsSchema(district)
district

,name,layer,status,geometry
0,Бокситогорский муниципальный район,Бокситогорский муниципальный район,district,"MULTIPOLYGON (((34.32834 59.19564, 34.32777 59..."
1,Кингисеппский муниципальный район,Кингисеппский муниципальный район,district,"MULTIPOLYGON (((26.67752 59.96863, 26.67743 59..."
2,Киришский муниципальный район,Киришский муниципальный район,district,"MULTIPOLYGON (((31.94177 59.42479, 31.94327 59..."
3,Кировский муниципальный район,Кировский муниципальный район,district,"MULTIPOLYGON (((31.91715 59.95987, 31.91722 59..."
4,Лодейнопольский муниципальный район,Лодейнопольский муниципальный район,district,"MULTIPOLYGON (((32.95801 60.71431, 33.02438 60..."
5,Ломоносовский муниципальный район,Ломоносовский муниципальный район,district,"MULTIPOLYGON (((28.84956 59.78505, 28.84950 59..."
6,Подпорожский муниципальный район,Подпорожский муниципальный район,district,"MULTIPOLYGON (((35.52906 60.91530, 35.52806 60..."
7,Приозерский муниципальный район,Приозерский муниципальный район,district,"MULTIPOLYGON (((30.69201 60.50326, 30.68940 60..."
8,Сосновоборский городской округ,Сосновоборский городской округ,district,"MULTIPOLYGON (((28.97562 59.81609, 28.97583 59..."
9,Волосовский муниципальный район,Волосовский муниципальный район,district,"MULTIPOLYGON (((28.98894 59.48069, 28.98604 59..."


In [ ]:
settlement = IndicatorsSchema(settlement)
settlement

,name,layer,status,geometry
0,Самойловское сельское поселение,Бокситогорский муниципальный район,settlement,"MULTIPOLYGON (((34.42169 59.68391, 34.42428 59..."
1,Большедворское сельское поселение,Бокситогорский муниципальный район,settlement,"MULTIPOLYGON (((34.30682 59.71494, 34.30791 59..."
2,Пикалевское городское поселение,Бокситогорский муниципальный район,settlement,"MULTIPOLYGON (((34.14251 59.48759, 34.13616 59..."
3,Борское сельское поселение,Бокситогорский муниципальный район,settlement,"MULTIPOLYGON (((34.05997 59.15087, 34.06007 59..."
4,Бокситогорское городское поселение,Бокситогорский муниципальный район,settlement,"MULTIPOLYGON (((34.06836 59.40315, 34.05535 59..."
...,...,...,...,...
183,Фёдоровское городское поселение,Тосненский муниципальный район,settlement,"MULTIPOLYGON (((30.52049 59.69087, 30.52316 59..."
184,Нурминское сельское поселение,Тосненский муниципальный район,settlement,"MULTIPOLYGON (((31.01271 59.63362, 31.01394 59..."
185,Красноборское городское поселение,Тосненский муниципальный район,settlement,"MULTIPOLYGON (((30.58336 59.64993, 30.58367 59..."
186,Трубникоборское сельское поселение,Тосненский муниципальный район,settlement,"MULTIPOLYGON (((31.13126 59.08942, 31.13522 59..."


In [ ]:
gdfs = [city,district,settlement]

In [ ]:
gdfs = [IndicatorsSchema(gdf) for gdf in gdfs] 

In [ ]:
gdfs[1]

,name,layer,status,geometry
0,Бокситогорский муниципальный район,Бокситогорский муниципальный район,district,"MULTIPOLYGON (((34.32834 59.19564, 34.32777 59..."
1,Кингисеппский муниципальный район,Кингисеппский муниципальный район,district,"MULTIPOLYGON (((26.67752 59.96863, 26.67743 59..."
2,Киришский муниципальный район,Киришский муниципальный район,district,"MULTIPOLYGON (((31.94177 59.42479, 31.94327 59..."
3,Кировский муниципальный район,Кировский муниципальный район,district,"MULTIPOLYGON (((31.91715 59.95987, 31.91722 59..."
4,Лодейнопольский муниципальный район,Лодейнопольский муниципальный район,district,"MULTIPOLYGON (((32.95801 60.71431, 33.02438 60..."
5,Ломоносовский муниципальный район,Ломоносовский муниципальный район,district,"MULTIPOLYGON (((28.84956 59.78505, 28.84950 59..."
6,Подпорожский муниципальный район,Подпорожский муниципальный район,district,"MULTIPOLYGON (((35.52906 60.91530, 35.52806 60..."
7,Приозерский муниципальный район,Приозерский муниципальный район,district,"MULTIPOLYGON (((30.69201 60.50326, 30.68940 60..."
8,Сосновоборский городской округ,Сосновоборский городской округ,district,"MULTIPOLYGON (((28.97562 59.81609, 28.97583 59..."
9,Волосовский муниципальный район,Волосовский муниципальный район,district,"MULTIPOLYGON (((28.98894 59.48069, 28.98604 59..."


### Валидация для графа

In [4]:
citygraph = graphbuilder.get_graph_from_polygon(city, crs=local_crs,country_polygon=russia)

/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/osmnx/_overpass.py:254: UserWarning: This area is 52 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


In [6]:
citygraph = graphbuilder.convert_list_attr_from_str(citygraph)

In [7]:
citygraph = indicators.prepare_graph(citygraph)

In [8]:
citygraph_n,citygraph_e = momepy.nx_to_gdf(citygraph)

In [27]:
citygraph_n

,reg_1,reg_2,x,y,nodeID,exit,exit_country,geometry
0,False,False,316822.011829,6.514139e+06,0,NaN,NaN,POINT (316822.012 6514139.318)
1,False,False,316864.213100,6.514292e+06,1,NaN,NaN,POINT (316864.213 6514292.399)
2,False,False,317015.043435,6.514061e+06,2,NaN,NaN,POINT (317015.043 6514060.895)
3,False,False,316716.593408,6.514168e+06,3,NaN,NaN,POINT (316716.593 6514168.399)
4,False,False,316771.697024,6.513968e+06,4,NaN,NaN,POINT (316771.697 6513968.255)
...,...,...,...,...,...,...,...,...
104065,False,False,222720.907545,6.728057e+06,104065,NaN,NaN,POINT (222720.908 6728057.151)
104066,False,False,222481.292591,6.728106e+06,104066,NaN,NaN,POINT (222481.293 6728106.467)
104067,False,False,222429.409497,6.727811e+06,104067,NaN,NaN,POINT (222429.409 6727810.800)
104068,False,False,222453.683954,6.727778e+06,104068,NaN,NaN,POINT (222453.684 6727777.907)


In [44]:
citygraph_e

,length_meter,geometry,type,time_sec,time_min,highway,maxspeed,reg,ref,is_exit,node_start,node_end
0,158.791,"LINESTRING (316822.012 6514139.318, 316864.213...",car,9.527,0.158783,tertiary,16.666667,3,NaN,0,0,1
1,209.104,"LINESTRING (316822.012 6514139.318, 316878.621...",car,12.546,0.209100,tertiary,16.666667,3,NaN,0,0,2
2,109.356,"LINESTRING (316822.012 6514139.318, 316793.074...",car,6.561,0.109350,tertiary,16.666667,3,NaN,0,0,3
3,178.371,"LINESTRING (316822.012 6514139.318, 316821.070...",car,10.702,0.178367,residential,16.666667,3,NaN,0,0,4
4,166.563,"LINESTRING (316864.213 6514292.399, 316905.768...",car,9.994,0.166567,tertiary,16.666667,3,NaN,0,1,284
...,...,...,...,...,...,...,...,...,...,...,...,...
263693,42.975,"LINESTRING (222429.409 6727810.800, 222434.555...",car,2.579,0.042983,unclassified,16.666667,3,NaN,0,104067,104068
263694,443.462,"LINESTRING (222453.684 6727777.907, 222478.142...",car,26.608,0.443467,unclassified,16.666667,3,NaN,0,104068,104068
263695,443.462,"LINESTRING (222453.684 6727777.907, 222468.710...",car,26.608,0.443467,unclassified,16.666667,3,NaN,0,104068,104068
263696,42.975,"LINESTRING (222453.684 6727777.907, 222434.555...",car,2.579,0.042983,unclassified,16.666667,3,NaN,0,104068,104067


In [16]:
citygraph_e.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 263698 entries, 0 to 263697
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   length_meter  263698 non-null  float64 
 1   geometry      263698 non-null  geometry
 2   type          263698 non-null  object  
 3   time_sec      263698 non-null  float64 
 4   time_min      263698 non-null  float64 
 5   highway       263698 non-null  object  
 6   maxspeed      263698 non-null  float64 
 7   reg           263698 non-null  int64   
 8   ref           27016 non-null   object  
 9   is_exit       263698 non-null  int64   
 10  node_start    263698 non-null  int64   
 11  node_end      263698 non-null  int64   
dtypes: float64(4), geometry(1), int64(4), object(3)
memory usage: 24.1+ MB


In [10]:
# классы валидации графа в виде гдф

class GraphNode(BaseSchema):
    
    reg_1: Series[bool]
    reg_2: Series[bool]
    x: Series[float]
    y: Series[float]
    nodeID: Series[int]
    exit: Series[float] = pa.Field(nullable=True)
    exit_country: Series[float] = pa.Field(nullable=True)
    _geom_types = [Point]
    

class GraphEdge(BaseSchema):
    
    length_meter: Series[float]
    type: Series[str]
    time_sec: Series[float]
    time_min: Series[float]
    highway: Series[str] = pa.Field(coerce=True)
    maxspeed: Series[float]
    reg: Series[int]
    ref: Series[str] = pa.Field(nullable=True,coerce=True)
    is_exit: Series[int]
    node_start: Series[int]
    node_end: Series[int]
    _geom_types = [LineString, MultiLineString]

In [11]:
GraphNode(citygraph_n)


,reg_1,reg_2,x,y,nodeID,exit,exit_country,geometry
0,False,False,316822.011829,6.514139e+06,0,NaN,NaN,POINT (316822.012 6514139.318)
1,False,False,316864.213100,6.514292e+06,1,NaN,NaN,POINT (316864.213 6514292.399)
2,False,False,317015.043435,6.514061e+06,2,NaN,NaN,POINT (317015.043 6514060.895)
3,False,False,316716.593408,6.514168e+06,3,NaN,NaN,POINT (316716.593 6514168.399)
4,False,False,316771.697024,6.513968e+06,4,NaN,NaN,POINT (316771.697 6513968.255)
...,...,...,...,...,...,...,...,...
126388,False,False,222720.907545,6.728057e+06,126388,NaN,NaN,POINT (222720.908 6728057.151)
126389,False,False,222481.292591,6.728106e+06,126389,NaN,NaN,POINT (222481.293 6728106.467)
126390,False,False,222429.409497,6.727811e+06,126390,NaN,NaN,POINT (222429.409 6727810.800)
126391,False,False,222453.683954,6.727778e+06,126391,NaN,NaN,POINT (222453.684 6727777.907)


In [12]:
GraphEdge(citygraph_e)


,length_meter,geometry,type,time_sec,time_min,highway,maxspeed,reg,ref,is_exit,node_start,node_end
0,158.791,"LINESTRING (316822.012 6514139.318, 316864.213...",car,9.527,0.158783,tertiary,16.666667,3,NaN,0,0,1
1,209.104,"LINESTRING (316822.012 6514139.318, 316878.621...",car,12.546,0.209100,tertiary,16.666667,3,NaN,0,0,2
2,109.356,"LINESTRING (316822.012 6514139.318, 316793.074...",car,6.561,0.109350,tertiary,16.666667,3,NaN,0,0,3
3,178.371,"LINESTRING (316822.012 6514139.318, 316821.070...",car,10.702,0.178367,residential,16.666667,3,NaN,0,0,4
4,166.563,"LINESTRING (316864.213 6514292.399, 316905.768...",car,9.994,0.166567,tertiary,16.666667,3,NaN,0,1,284
...,...,...,...,...,...,...,...,...,...,...,...,...
313586,42.975,"LINESTRING (222429.409 6727810.800, 222434.555...",car,2.579,0.042983,unclassified,16.666667,3,NaN,0,126390,126391
313587,443.462,"LINESTRING (222453.684 6727777.907, 222478.142...",car,26.608,0.443467,unclassified,16.666667,3,NaN,0,126391,126391
313588,443.462,"LINESTRING (222453.684 6727777.907, 222468.710...",car,26.608,0.443467,unclassified,16.666667,3,NaN,0,126391,126391
313589,42.975,"LINESTRING (222453.684 6727777.907, 222434.555...",car,2.579,0.042983,unclassified,16.666667,3,NaN,0,126391,126390
